# 09 — Complete Experiment Story

**Built entirely from already-saved results.** This notebook makes zero new model/API
calls (one exception, clearly marked: it reads `results/runs/run_AV01_*` files if they exist yet -
that background run was launched separately and may still be in progress). Its job is not to
produce new evidence but to stitch every existing experiment into one coherent, honest narrative:
what we tried, what we learned, what we rejected, what we chose, and what remains uncertain.

Every number below is loaded live from its source file in the cells that follow - nothing is
hardcoded from memory. If a cited file is missing or a number can't be found, the cell says so
explicitly rather than guessing.

## 1. Problem and evaluation goal

Given an NDA document and one of 17 standard confidentiality requirements, classify whether the
NDA **Entails**, **Contradicts**, or does **Not Mention** that requirement, and return the exact
evidence clause the classification rests on (evidence grounding - not just a label).

Key metrics used throughout (full definitions: `docs/evaluation_protocol.md`):
- Accuracy, Macro-F1
- Contradiction recall (+ 95% CI) - the headline risk metric
- Joint label+evidence correctness - label right AND evidence right
- Cost (USD), latency (ms)

In [1]:
import json
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

def load_latest(path):
    p = REPO_ROOT / path
    if not p.exists():
        return None
    lines = [l for l in p.read_text().splitlines() if l.strip()]
    return json.loads(lines[-1]) if lines else None

def load_json(path):
    p = REPO_ROOT / path
    return json.loads(p.read_text()) if p.exists() else None

## 2. Dataset and split roles

| Data | Role | Reused adaptively? |
|---|---|---|
| 150-case dev sample (seed=42) | Oracle, model selection, RAG e2e, prompt tuning, confidence/abstention, agent experiments | **Yes** - same sample across nearly every tuning decision |
| Golden/regression cases (45) | Catch known behavioural regressions | Yes, by design (re-run after every change) |
| Robustness/security cases (46) | System behaviour (injection, error isolation, log hygiene) | Yes, by design |
| Architecture-validation set (340 cases, 20 untouched train-split documents) | One-time check of the frozen architecture on data no tuning decision ever saw | **No - built and run exactly once** (see Section 12) |
| Official test (the final, locked ContractNLI test-set evaluation) | Final, historical benchmark | No - run once, not re-tuned |

**This is disclosed explicitly, not glossed over**: the 150-case dev sample drove nearly every
architecture and hyperparameter decision in this project. Development-sample numbers below should
be read as exploratory evidence, not unbiased generalization estimates
(`docs/evaluation_protocol.md`).

In [2]:
val_report = load_json("data/validation_report.json")
print("=== Experiment 1: Data validation (A01-A07) ===")
if val_report:
    a1, a2, a3, a4, a5 = (val_report[k] for k in
        ["A01_schema_validation", "A02_missing_corrupt", "A03_duplicate_detection",
         "A04_leakage_test", "A05_label_distribution"])
    print(f"Parse success: {a1['parse_success_rate']:.1%} ({a1['total_parsed']}/{a1['total_raw']} docs)")
    print(f"Corrupt rate: {a2['corrupt_rate']:.1%}")
    print(f"Duplicate rate: {a3['duplicate_rate']:.2%} ({a3['duplicate_doc_count']} docs)")
    print(f"Split leakage (docs appearing in >1 split): {a4['leaked_doc_count']}")
    print(f"Label distribution (train): {a5['train']}")
else:
    print("data/validation_report.json not found")

=== Experiment 1: Data validation (A01-A07) ===
Parse success: 100.0% (607/607 docs)
Corrupt rate: 0.0%
Duplicate rate: 0.33% (2 docs)
Split leakage (docs appearing in >1 split): 0
Label distribution (train): {'Entailment': 3530, 'Contradiction': 841, 'NotMentioned': 2820}


**What this implied for architecture**: the true majority class is Entailment (~48.6% train),
not NotMentioned as the original planning document assumed - this corrected the majority-class
baseline's target label before any model was built. Zero split leakage confirmed at the document
level (verified independently again in Section 12 for the new architecture-validation split).

## Experiment 2 — Oracle / Full-Context ceiling

In [3]:
oracle = load_latest("results/archive/runs/run_B04_oracle_google_gemini-2.5-flash-lite.jsonl")
full_ctx = load_latest("results/archive/runs/run_B03_full_context.jsonl")
print("=== Oracle (gold evidence given directly) ===")
if oracle:
    m = oracle["metrics"]
    print(f"Accuracy: {m['accuracy']:.1%}  Macro-F1: {m['macro_f1']:.3f}  "
          f"Joint: {m.get('joint_label_evidence_correctness'):.3f} (should equal accuracy - it does)")
print("\n=== Full-context (whole document, no retrieval) - dev sample ===")
if full_ctx:
    m = full_ctx["metrics"]
    print(f"Accuracy: {m['accuracy']:.1%}  Macro-F1: {m['macro_f1']:.3f}  "
          f"Joint: {m.get('joint_label_evidence_correctness')} "
          f"(NOTE: this joint value is known-unreliable - see the flagged caveat below)")

=== Oracle (gold evidence given directly) ===
Accuracy: 95.3%  Macro-F1: 0.948  Joint: 0.953 (should equal accuracy - it does)

=== Full-context (whole document, no retrieval) - dev sample ===
Accuracy: 91.3%  Macro-F1: 0.871  Joint: 0.36 (NOTE: this joint value is known-unreliable - see the flagged caveat below)


**What this showed**: Oracle's 95.3% accuracy means the model reasons well when handed
unambiguous evidence directly - the real bottleneck is finding evidence, not reasoning about it.
Full-context (91.3% dev-sample accuracy) beats every retrieval-based architecture tested in this
project on raw accuracy, which is the single most important, least comfortable finding in the whole
project (Section 12).

**A caveat found during verification, disclosed here rather than hidden**:
`scripts/run_full_context_baseline.py` (source of the number above) never populated
`retrieved_span_indices`, the same bug independently found and fixed in the official test-set evaluation script
(see the joint-metric bug entry in `docs/decisions.md`). That means this run's `joint_label_evidence_correctness` value is
not trustworthy (full-context should have joint == accuracy by construction, since it sees the
whole document) - **only the accuracy/macro-F1 numbers above are reliable for this specific
result file.**

## Experiment 3 — Model selection

In [4]:
gpt5 = load_latest("results/archive/runs/run_B04_oracle_openai_gpt-5-mini.jsonl")
print("=== Model bake-off (Oracle, 150-case sample) ===")
for name, rec in [("openai/gpt-5-mini", gpt5), ("google/gemini-2.5-flash-lite", oracle)]:
    if rec:
        m = rec["metrics"]
        cost = sum((p.get("cost_latency") or {}).get("cost_usd", 0) or 0 for p in rec["predictions"])
        print(f"{name:32s} acc={m['accuracy']:.3f} macro_f1={m['macro_f1']:.3f} cost=${cost:.4f}")

=== Model bake-off (Oracle, 150-case sample) ===
openai/gpt-5-mini                acc=0.967 macro_f1=0.951 cost=$0.1549
google/gemini-2.5-flash-lite     acc=0.953 macro_f1=0.948 cost=$0.0125


**Decision**: switched to `google/gemini-2.5-flash-lite` - 12.4x cheaper, 6x faster, ~1.4pt
lower accuracy, identical risk-sensitive recall (1.000 both). Full rationale: see the model-choice decision in `docs/decisions.md`.

## Experiment 4 — Retrieval study

In [5]:
print("=== Retrieval configuration rounds (all local, zero LLM cost) ===")
retrieval_files = {
    "Round 1 (chunk method/size baseline)": "data/retrieval_experiment_results.json",
    "Rule vs semantic": "data/rule_vs_semantic_retrieval.json",
    "Round 3 (reranking)": "data/reranking_comparison.json",
    "Round 4 (embeddings/hybrid/reranker size)": "data/full_retrieval_comparison.json",
    "Round 5 (top-K sweep)": "data/top_k_sweep.json",
    "Round 6 (candidate pool sweep)": "data/pool_size_sweep.json",
    "Round 7 (rule-boost RRF, ADOPTED)": "data/rule_boosted_retrieval.json",
    "Round 8 (parent-child, REJECTED)": "data/parent_child_retrieval.json",
    "Round 9 (overlapping windows, REJECTED)": "data/overlapping_chunks_comparison.json",
    "Round 10 (stronger rerankers, REJECTED)": "data/stronger_reranker_comparison.json",
}
for name, path in retrieval_files.items():
    d = load_json(path)
    print(f"{name}: {'found' if d is not None else 'MISSING'} -> {path}")

print("\nFinal adopted config (docs/decisions.md ADR-002): sentence chunking -> mpnet embeddings ->")
print("retrieve top-20 -> rerank ms-marco-MiniLM-L-12-v2 -> keep top-7 -> RRF-fuse rule match.")
print("\nRound 7 (adopted) result:")
r7 = load_json("data/rule_boosted_retrieval.json")
if r7:
    print(json.dumps(r7, indent=2)[:600])

=== Retrieval configuration rounds (all local, zero LLM cost) ===
Round 1 (chunk method/size baseline): found -> data/retrieval_experiment_results.json
Rule vs semantic: found -> data/rule_vs_semantic_retrieval.json
Round 3 (reranking): found -> data/reranking_comparison.json
Round 4 (embeddings/hybrid/reranker size): found -> data/full_retrieval_comparison.json
Round 5 (top-K sweep): found -> data/top_k_sweep.json
Round 6 (candidate pool sweep): found -> data/pool_size_sweep.json
Round 7 (rule-boost RRF, ADOPTED): found -> data/rule_boosted_retrieval.json
Round 8 (parent-child, REJECTED): found -> data/parent_child_retrieval.json
Round 9 (overlapping windows, REJECTED): found -> data/overlapping_chunks_comparison.json
Round 10 (stronger rerankers, REJECTED): found -> data/stronger_reranker_comparison.json

Final adopted config (docs/decisions.md ADR-002): sentence chunking -> mpnet embeddings ->
retrieve top-20 -> rerank ms-marco-MiniLM-L-12-v2 -> keep top-7 -> RRF-fuse rule match.

R

**Failed alternatives, preserved not hidden**: parent-child expansion and overlapping windows
both raised recall but collapsed MRR (the ranking-quality metric) - a real regression, not a
trade-off, so both were rejected. A stronger cross-encoder reranker was also tried and lost on all
three metrics. Full numbers: see the retrieval-configuration decision in `docs/decisions.md`.

## Experiment 5 — RAG end-to-end

In [6]:
rag_e2e = load_latest("results/archive/runs/run_T024_rag.jsonl")
print("=== RAG end-to-end (T024), 150-case dev sample ===")
if rag_e2e:
    m = rag_e2e["metrics"]
    print(f"Accuracy: {m['accuracy']:.1%} (Oracle ceiling: 95.3%, Full-context: 91.3%)")
    print(f"Macro-F1: {m['macro_f1']:.3f}  Joint: {m.get('joint_label_evidence_correctness'):.3f}")
print("\nDominant failure mode (from docs/decisions.md ADR-003): NotMentioned falsely predicted as")
print("Entailment/Contradiction - 12 of 21 errors (57%). This directly motivated prompt v2 (Exp. 6).")

=== RAG end-to-end (T024), 150-case dev sample ===
Accuracy: 86.0% (Oracle ceiling: 95.3%, Full-context: 91.3%)
Macro-F1: 0.842  Joint: 0.813

Dominant failure mode (from docs/decisions.md ADR-003): NotMentioned falsely predicted as
Entailment/Contradiction - 12 of 21 errors (57%). This directly motivated prompt v2 (Exp. 6).


## Experiment 6 — Prompt iterations (v1 -> v6)

In [7]:
prompt_files = {
    "v2 (adopted over v1)": "results/archive/runs/run_T018_prompt_v2.jsonl",
    "v3 (rejected)": "results/archive/runs/run_T018_prompt_v3.jsonl",
    "v4 (rejected)": "results/archive/runs/run_T018_prompt_v4.jsonl",
    "v5 (tested, not adopted)": "results/archive/runs/run_T018_prompt_v5.jsonl",
    "v6 (current default)": "results/archive/runs/run_T018_prompt_v6.jsonl",
}
print("=== Prompt version comparison, identical 150-case sample ===")
print(f"{'Version':30s} {'Acc':>7s} {'MacroF1':>8s} {'ContraRecall':>13s}")
for name, path in prompt_files.items():
    rec = load_latest(path)
    if rec:
        m = rec["metrics"]
        cr = m.get("contradiction_recall")
        print(f"{name:30s} {m['accuracy']:7.1%} {m['macro_f1']:8.3f} "
              f"{(f'{cr:.1%}' if cr is not None else 'n/a'):>13s}")

=== Prompt version comparison, identical 150-case sample ===
Version                            Acc  MacroF1  ContraRecall
v2 (adopted over v1)             88.0%    0.858         85.7%
v3 (rejected)                    84.7%    0.803         64.3%
v4 (rejected)                    84.7%    0.696         21.4%
v5 (tested, not adopted)         86.0%    0.850         85.7%
v6 (current default)             89.3%    0.864         78.6%


**Critical timing note, corrected after a forensic timestamp/git review** (see
`docs/decisions.md` ADR-010): the official test evaluation ran in two genuinely different
configurations, not one run at two sample sizes. An interim 500-case run used prompt v2; the full
2,091-case run - the numbers cited everywhere as "the official test result" - used v6, adopted the
the same commit (`dd797d1`) that also replaced RAG+agent's routing
logic with the decoupled fix. All three full-2,091-case runs happened a clear, multi-hour gap after that
commit (checked directly against each run's own saved timestamp and the commit's own timestamp,
both still present in the underlying files/git history even though not quoted here). **The full-scale numbers are already v6 and already use the decoupled routing fix - they
are not stale relative to what currently ships.** See Section 16 for the full two-phase breakdown.

## Experiment 7 — Confidence / Abstention

In [8]:
conf = load_json("data/confidence_analysis.json")
print("=== Confidence/routing signal comparison ===")
if conf:
    print(json.dumps(conf, indent=2)[:800])
else:
    print("data/confidence_analysis.json not found")

=== Confidence/routing signal comparison ===
{
  "f01_self_confidence_auroc": 0.5542929292929293,
  "f01_calibration": [
    {
      "bin": "[0.00, 0.50)",
      "n": 8,
      "avg_confidence": 0.0,
      "empirical_accuracy": 1.0
    },
    {
      "bin": "[0.70, 0.85)",
      "n": 4,
      "avg_confidence": 0.8,
      "empirical_accuracy": 0.25
    },
    {
      "bin": "[0.85, 0.95)",
      "n": 6,
      "avg_confidence": 0.9,
      "empirical_accuracy": 0.8333333333333334
    },
    {
      "bin": "[0.95, 1.01)",
      "n": 132,
      "avg_confidence": 1.0,
      "empirical_accuracy": 0.8939393939393939
    }
  ],
  "f02_retrieval_score_auroc": 0.4638047138047138,
  "extra_score_margin_auroc": 0.49074074074074076,
  "extra_rule_agrees_auroc": 0.6565656565656566,
  "best_signal": "rule_agrees",
  "best_signal_auroc": 0.65656565656


**Conclusion (see the confidence/abstention design decision in docs/decisions.md)**: every signal tested fell short of the 0.7 AUROC
target for hard abstention - self-confidence 0.554 (near chance, the model reports confidence=1.0
on 88% of cases regardless of correctness), retrieval score 0.464 (below chance), rule-agreement
best at 0.657-0.660. **Hard abstention was rejected**; the system routes ACCEPT/REVIEW instead of
ACCEPT/ABSTAIN. This is a genuinely weak routing signal, not a strong one - stated plainly rather
than oversold.

## Experiment 8 — Selective Agent

In [9]:
agent_exp = load_json("data/agent_experiment.json")
print("=== Agent experiment, 67 REVIEW-routed dev-sample cases ===")
if agent_exp:
    for k, v in agent_exp.items():
        if k != "outcomes":
            print(f"  {k}: {v}")

print("\n=== Re-verified against the FULL 2,091-case T041 hosted test set (notebooks/07) ===")
rag_full = load_latest("results/final/run_T041_final_test_rag_google_gemini-2.5-flash-lite.jsonl")
agent_full = load_latest("results/final/run_T041_final_test_rag_agent_google_gemini-2.5-flash-lite.jsonl")
if rag_full and agent_full:
    print(f"RAG accuracy:       {rag_full['metrics']['accuracy']:.1%} (n={rag_full['config'].get('sample_size')})")
    print(f"RAG+agent accuracy: {agent_full['metrics']['accuracy']:.1%} (n={agent_full['config'].get('sample_size')})")
    print("At full test-set scale, RAG+agent is numerically BELOW plain RAG - the dev-sample")
    print("finding reverses. See notebooks/07_selective_agent_experiments.ipynb for the full")
    print("McNemar analysis (p=0.088, not significant either direction) - this is disclosed as an")
    print("open, unresolved contradiction, not resolved in either direction here.")

=== Agent experiment, 67 REVIEW-routed dev-sample cases ===
  n_review_cases: 67
  rag_accuracy_on_review: 0.8059701492537313
  agent_accuracy_on_review: 0.8507462686567164
  n_recovery: 6
  n_regression: 3
  n_no_change: 58
  avg_steps: 1.3134328358208955
  stopped_reasons: {'concluded': 52, 'duplicate_loop': 14, 'invalid_action': 1}
  total_cost_usd: 0.016855400000000003
  overall_rag_correct: 132
  overall_with_agent: 135
  n_total_sample: 150

=== Re-verified against the FULL 2,091-case T041 hosted test set (notebooks/07) ===
RAG accuracy:       78.7% (n=2091)
RAG+agent accuracy: 77.7% (n=2091)
At full test-set scale, RAG+agent is numerically BELOW plain RAG - the dev-sample
finding reverses. See notebooks/07_selective_agent_experiments.ipynb for the full
McNemar analysis (p=0.088, not significant either direction) - this is disclosed as an
open, unresolved contradiction, not resolved in either direction here.


**Conclusion**: dev-sample McNemar p=0.51 (67 cases, not significant); full official-test scale
(T041-B, already v6 + decoupled routing - not a stale-configuration artifact) McNemar p=0.088 (152
discordant pairs, still not significant, but direction reversed - regression now exceeds recovery).
**Because T041-B already reflects the current shipped configuration, this reversal cannot be
attributed to "it was still running the old prompt/routing" - it wasn't.** The agent's benefit is
numerically promising on the small dev sample and numerically negative on the large, already-current
official-test sample - genuinely inconclusive, not proof of superiority in either direction. The
untouched architecture-validation run (Section 12b) provides a third, independent data point in the
same direction. Full detail: see the agent include/exclude decision in `docs/decisions.md`, and
`notebooks/07_selective_agent_experiments.ipynb`.

## 12. Architecture comparison

Same table as `notebooks/08_architecture_selection.ipynb`, reproduced here as the story's
centerpiece rather than re-derived differently - loaded fresh from the same source files.

In [10]:
def row_cost_latency(rec):
    preds = rec["predictions"]
    n = len(preds)
    total_cost = sum((p.get("cost_latency") or {}).get("cost_usd", 0) or 0 for p in preds)
    avg_lat = sum((p.get("cost_latency") or {}).get("latency_ms", 0) or 0 for p in preds) / n if n else 0
    return n, total_cost, (total_cost / n if n else 0), avg_lat

def print_table(title, rows):
    print(f"=== {title} ===")
    print(f"{'Architecture':15s} {'Prompt':8s} {'N (true)':>9s} {'Acc':>7s} {'MacroF1':>8s} "
          f"{'ContraRecall':>13s} {'TotalCost':>10s} {'Cost/case':>10s} {'AvgLatency':>11s}")
    for arch, path in rows:
        rec = load_latest(path)
        if not rec:
            print(f"{arch:15s} MISSING -> {path}")
            continue
        m = rec["metrics"]
        prompt_version = rec["config"].get("prompt_version") or "n/a"
        cr = m.get("contradiction_recall")
        n, total_cost, per_case, avg_lat = row_cost_latency(rec)
        config_n = rec["config"].get("sample_size")
        flag = "" if config_n == n else f"  *** config.sample_size={config_n} != len(predictions)={n} ***"
        print(f"{arch:15s} {prompt_version:8s} {n:9d} {m['accuracy']:7.1%} {m['macro_f1']:8.3f} "
              f"{(f'{cr:.1%}' if cr is not None else 'n/a'):>13s} "
              f"${total_cost:9.4f} ${per_case:9.6f} {avg_lat:9.0f}ms{flag}")
    print()

# --- Table 1: FINAL results - official test set, identical 2,091 cases, frozen v6 config ---
# N is always len(predictions), never config.sample_size (see the dev rule baseline check below
# for why that field cannot be trusted blindly - it shows config.sample_size=0 while the file
# actually has 1,037 real predictions).
table1_rows = [
    ("Rule",         "results/final/run_T041_final_test_rule_full.jsonl"),
    ("Full-context", "results/final/run_T041_final_test_full_context_google_gemini-2.5-flash-lite.jsonl"),
    ("RAG",          "results/final/run_T041_final_test_rag_google_gemini-2.5-flash-lite.jsonl"),
    ("RAG + agent",  "results/final/run_T041_final_test_rag_agent_google_gemini-2.5-flash-lite.jsonl"),
]
print_table("Table 1 - FINAL RESULTS (official test set, identical 2,091 cases, frozen v6 config)", table1_rows)

print("*** COST DIRECTLY SHAPED THE ARCHITECTURE CHOICE: RAG+agent costs ~2.7x plain RAG per case")
print("*** (two classifier calls for routing independence, plus the agent's own calls on REVIEW")
print("*** cases) and full-context costs ~2.2x RAG per case (no retrieval, but the whole document")
print("*** goes to the model every time). All three are still cheap in absolute terms at this")
print("*** document length (well under $0.001/case even for the most expensive architecture) - see")
print("*** docs/decisions.md's full-context-exclusion decision for why cost alone did NOT settle")
print("*** this (the real objection is long-document scalability, not the dollar amount measured here).")
print()

# --- Table 2: DEVELOPMENT HISTORY - 150-case reused tuning sample. DO NOT COMPARE TO TABLE 1. ---
table2_rows = [
    ("Rule",         "results/archive/runs/run_B02_rule_baseline.jsonl"),
    ("Full-context", "results/archive/runs/run_B03_full_context.jsonl"),
    ("RAG",          "results/archive/runs/run_T018_prompt_v2.jsonl"),
]
print_table("Table 2 - DEVELOPMENT HISTORY (150-case reused tuning sample - DO NOT COMPARE TO TABLE 1)", table2_rows)

# RAG + agent on the dev sample isn't its own standalone result file - it's RAG's own dev-sample
# cost/predictions PLUS the agent's incremental cost on the 67 REVIEW-routed cases
# (data/agent_experiment.json). Composed explicitly here rather than left out of the table.
rag_dev = load_latest("results/archive/runs/run_T018_prompt_v2.jsonl")
agent_exp = load_json("data/agent_experiment.json")
if rag_dev and agent_exp:
    _, rag_cost, _, rag_lat = row_cost_latency(rag_dev)
    combined_cost = rag_cost + agent_exp["total_cost_usd"]
    acc = agent_exp["overall_with_agent"] / agent_exp["n_total_sample"]
    print(f"{'RAG + agent':15s} {'v2':8s} {150:9d} {acc:7.1%} {'n/a':>8s} "
          f"{'n/a':>13s} ${combined_cost:9.4f} ${combined_cost/150:9.6f} {'n/a':>11s}")
    print("  (^ RAG's own dev-sample cost + the agent's incremental cost on 67 REVIEW-routed cases -")
    print("     composed from two files, not a single saved record; latency wasn't tracked per-case")
    print("     in the agent experiment's saved output, so it's left as n/a rather than guessed.")
    print("     True N is 150 - this is the SAME 150-case dev sample as the other Table 2 rows.)")

print("\n*** Table 2 reuses the SAME adaptively-tuned 150-case sample used for every earlier decision")
print("*** in this notebook. This is development evidence, not clean held-out validation - see")
print("*** docs/evaluation_protocol.md. Table 2's RAG row also used prompt v2, not the current v6")
print("*** default - another reason not to compare it directly to Table 1's v6 numbers.")


=== Table 1 - FINAL RESULTS (official test set, identical 2,091 cases, frozen v6 config) ===
Architecture    Prompt    N (true)     Acc  MacroF1  ContraRecall  TotalCost  Cost/case  AvgLatency
Rule            n/a           2091   59.0%    0.479         16.8% $   0.0000 $ 0.000000         0ms
Full-context    v6            2091   81.2%    0.760         59.1% $   0.6851 $ 0.000328      1111ms
RAG             v6            2091   78.7%    0.738         63.6% $   0.3169 $ 0.000152      1032ms
RAG + agent     v6            2091   77.7%    0.727         60.5% $   0.8477 $ 0.000405      2671ms

*** COST DIRECTLY SHAPED THE ARCHITECTURE CHOICE: RAG+agent costs ~2.7x plain RAG per case
*** (two classifier calls for routing independence, plus the agent's own calls on REVIEW
*** cases) and full-context costs ~2.2x RAG per case (no retrieval, but the whole document
*** goes to the model every time). All three are still cheap in absolute terms at this
*** document length (well under $0.001/case even

## Experiment 12b — Architecture validation (the untouched check)

In [11]:
print("=== Untouched architecture-validation run (340 cases, 20 never-before-used train-split documents) ===")
av_files = {
    "full_context": "results/final/run_AV01_architecture_validation_full_context.jsonl",
    "rag": "results/final/run_AV01_architecture_validation_rag.jsonl",
    "rag_agent": "results/final/run_AV01_architecture_validation_rag_agent.jsonl",
}
any_found = False
for arch, path in av_files.items():
    rec = load_latest(path)
    if rec:
        any_found = True
        m = rec["metrics"]
        print(f"{arch:15s} acc={m['accuracy']:.1%} macro_f1={m['macro_f1']:.3f} "
              f"contra_recall={m.get('contradiction_recall')} joint={m.get('joint_label_evidence_correctness')}")
    else:
        print(f"{arch:15s} NOT YET AVAILABLE -> {path}")

paired = load_json("data/architecture_validation_paired_comparison.json")
if paired:
    print("\nPaired comparison / McNemar:")
    print(json.dumps(paired, indent=2))
elif not any_found:
    print("\n*** STATUS AS OF THIS NOTEBOOK BUILD: the architecture-validation run (AV01) was")
    print("*** launched in the background and had not yet completed when this notebook was built.")
    print("*** Re-run this cell (or re-execute this notebook) once results/runs/run_AV01_*.jsonl")
    print("*** files exist to see the real, untouched-data comparison. Manifest: ")
    print("*** data/architecture_validation_manifest.json (340 cases, 20 documents, seed=99,")
    print("*** verified zero overlap with dev/test/golden-case doc_ids).")


=== Untouched architecture-validation run (340 cases, 20 never-before-used train-split documents) ===
full_context    acc=80.6% macro_f1=0.759 contra_recall=0.6216216216216216 joint=0.8058823529411765
rag             acc=80.3% macro_f1=0.764 contra_recall=0.6486486486486487 joint=0.7823529411764706
rag_agent       acc=77.6% macro_f1=0.722 contra_recall=0.5135135135135135 joint=0.7529411764705882

Paired comparison / McNemar:
{
  "full_context_vs_rag": {
    "both_correct": 255,
    "both_wrong": 48,
    "full_context_only_correct": 19,
    "rag_only_correct": 18,
    "mcnemar": {
      "n_common": 340,
      "b_a_only_correct": 19,
      "c_b_only_correct": 18,
      "n_discordant": 37,
      "p_value": 1.0,
      "significant_at_0.05": false
    }
  },
  "rag_vs_rag_agent": {
    "both_correct": 254,
    "both_wrong": 57,
    "rag_only_correct": 19,
    "rag_agent_only_correct": 10,
    "mcnemar": {
      "n_common": 340,
      "b_a_only_correct": 19,
      "c_b_only_correct": 10,
   

## 13. Architecture decision

**Frozen** (see the final architecture-freeze decision in `docs/decisions.md`): RAG + selective agent, based on the dev-sample
comparison above. Rule-based and full-context excluded on principle (full-context's cost/latency
and accuracy-dilution risk are expected to worsen on longer, real-world documents - a design
hypothesis, not yet validated by a long-document stress test). Between RAG and RAG+agent, the agent
was included for a dev-sample-measured +2pt accuracy gain at negligible cost.

**What was NOT validated at freeze time, and what has since been checked**: no evidence independent
of the reused dev sample existed until the untouched architecture-validation run (AV01, Section
12b) completed. Its result: full-context (80.6%) and RAG (80.3%) are statistically
indistinguishable (McNemar p=1.000), and RAG+agent (77.6%) underperforms both, correcting 10 routed
errors while introducing 19 - a net negative, consistent with the T041-B (full official-test scale)
finding above. **The freeze is not being reversed as part of this notebook, but the empirical case
for including the agent specifically is now weaker across three independent checks** (dev sample,
T041-B, and AV01), not just one.

## 14. Final architecture (reconstructed from code, not memory)

```text
POST /review -> review_document() [pipeline/orchestrator.py]
  per hypothesis -> review_requirement():
    sentence chunking -> dense retrieval (top-20) -> rerank (top-7)
    rule_result = classify_by_keywords()
    PATH A (rule-boosted): query_rerank_and_boost() -> classify() -> rag_result [returned if ACCEPT]
    PATH B (plain):        query_and_rerank()       -> classify() -> plain_result [routing signal only]
    route(confidence=rag_result.confidence, rule_agrees=(rule_result == plain_result.label))
      ACCEPT -> return rag_result
      REVIEW -> run_agent(...) -> return agent_result
```
Verified directly against `pipeline/orchestrator.py` in `docs/architecture.md` - two classifier
calls per requirement, not one, to keep the routing signal decoupled from the rule-boosted
retrieval it judges (see the routing-independence fix in `docs/decisions.md`).

## 15. Regression / robustness testing (kept separate from accuracy claims)

| Category | Cases | Real result | Counts toward accuracy benchmark? |
|---|---|---|---|
| Golden/ordinary | 30 | 24/30 = 80.0% | No - regression check only |
| Negative/wrong-behaviour | 15 | 10/15 = 66.7%, found a real exception-clause weakness (4/4 failed) | No - regression check only |
| Injection/security | 11 | 11/11 resisted (v6) | No - robustness check only |
| LLM behaviour | 10 | 10/10 | No - robustness check only |
| Agent behaviour | 10 | consistent with the agent experiment above | No - robustness check only |
| Data leakage prevention | 21 pytest tests | all pass | No - static/code check |
| API/error handling | 5 (partly code tests) | all pass | No - system check |
| Logging/security | 5 | 3/5 pass, 2/5 correctly blocked (per-request IDs not built) | No - system check |

Full detail and category definitions: `docs/evaluation_case_design.md`.

## 16. Official ContractNLI test-set evaluation - two distinct configurations (T041-A, T041-B), historical, with explicit caveats

**Corrected**: this evaluation is not one run at two sample sizes. A forensic review of
saved timestamps against git history found two genuinely different configurations:

- **T041-A** (interim, 500-case stratified subsample, run first): prompt v2 /
  `agent_step_v1.txt`; RAG+agent used the original inline, circular routing signal (pre-C1-fix).
- **T041-B** (the full 2,091-case test set, run later - **these are the numbers
  headlined everywhere as "the official test result"**): prompt v6 / `agent_step_v2.txt` (the
  current shipped default); RAG+agent used the decoupled routing fix via
  `pipeline/orchestrator.py::review_requirement()`.

Both changes landed together in the same commit (`dd797d1`), a clear, multi-hour gap before
every T041-B run (checked directly against each run's own saved timestamp and the commit's own
timestamp, both still present in the underlying files/git history even though not quoted here). **A previously-stated caveat in this notebook and in `docs/decisions.md`/
`docs/evaluation_protocol.md` was wrong and is retracted**: "T041 used prompt v2, not the current
v6 default" was true only for T041-A. T041-B's numbers are already v6 and already reflect the
decoupled routing fix - the RAG+agent underperformance on T041-B cannot be explained away as a
stale configuration.

Not superseded by anything in this notebook, including the untouched architecture-validation run
above - T041-B remains the official, historical ContractNLI test-split evaluation. Caveats that
actually apply to T041-B:

1. **The architecture decision this evaluation covers came from the reused dev sample** (Section
   12/13) - it measures the frozen architecture's real held-out performance, but does not
   retroactively validate that the *freeze itself* was well-supported.
2. **T041-B is not a pristine first exposure to the test split.** T041-A had already scored a
   500-case subsample of the same split before T041-B ran. The v2->v6 and routing changes were
   triggered by an independently discovered live security issue and a code-audit finding, not by
   looking at T041-A's scores - this is not test-set tuning in the overfitting sense, but the split
   had genuinely been partially observed before T041-B's numbers were produced.
3. **T041-A and T041-B's RAG+agent results are different configurations, not a sample-size
   effect** - comparing them directly (as earlier analyses in this project did) conflates a prompt
   change with a routing-logic change.
4. **The joint label+evidence correctness metric was broken in the code that executed both
   phases** (see the joint-metric bug entry in `docs/decisions.md`). Verified: only
   the hosted full-context T041-B result (full 2,091 cases) has a corrected, trustworthy joint value
   (0.812) - and that correction is itself a **post-hoc backfilled metric**
   (`scripts/backfill_joint_metric.py`), not something the original run computed correctly. Every
   other official-test joint value should be treated as unverified until backfilled.

## 17. What is scientifically established

- The model reasons well given clean evidence (Oracle 95.3%) - the real bottleneck is retrieval,
  not reasoning.
- `google/gemini-2.5-flash-lite` beats `gpt-5-mini` on quality-per-dollar with identical
  risk-sensitive recall.
- The tuned retrieval configuration (sentence chunking, mpnet, rerank L-12, top-7, rule-boost RRF)
  beats every tested alternative on recall/precision/MRR simultaneously (parent-child, overlap, and
  stronger rerankers were tried and lost).
- **Full-context is highly competitive and, on the dev sample and the full hosted official test set,
  has the single highest raw accuracy of any architecture tested.**
- The confidence/routing signal is weak (AUROC 0.657-0.660) - hard abstention was correctly
  rejected rather than oversold.
- A real prompt-injection vulnerability existed and was found and fixed (v6) at zero accuracy cost.
- A real, systematic weakness exists in exception/carve-out clause reconciliation (4/4 failures).

## 18. What is not yet established

- Whether RAG's advantage over full-context actually materializes on longer, real-world documents -
  no long-document stress test has been run (`docs/architecture.md`'s proposed, not-yet-run
  experiment).
- Whether the selective agent provides a real, positive effect - the evidence reverses direction
  between the dev sample and the full official-test scale (T041-B), and is never statistically
  significant across three independent checks (dev sample, T041-B, and the untouched AV01
  validation run).
- Whether a differently-configured agent (different routing threshold, different tool set) would
  perform better - AV01 and T041-B both test only the current frozen configuration.

**Resolved since this notebook was first built, no longer an open question**: whether the current
v6 shipped configuration would score differently from the official test run's numbers - it does not
need to, because the full 2,091-case official test run (T041-B) already used v6 and the decoupled
routing fix throughout (`docs/decisions.md` ADR-010).

## 19. Recommended next experiment - STATUS: IN PROGRESS, not merely proposed

The untouched architecture-validation experiment (Full-context vs. RAG vs. RAG+agent, run once, no
tuning after) was built and launched (`scripts/build_architecture_validation_set.py`,
`scripts/run_architecture_validation.py`). See Section 12b above for its current status - re-run
that cell (or this whole notebook) to pick up the real result once it completes. The remaining
open question after that (long-document scalability) is still genuinely unstarted - see
`docs/architecture.md`'s proposed stress test.

## 20. Final takeaway

**What we learned**: retrieval, not reasoning, is the real bottleneck; a cheaper, faster model
matched the more expensive one on the metric that matters most; full-context is a stronger or
statistically-tied competitor to the shipped architecture on every accuracy measurement taken so
far, including the untouched validation run; the confidence signal and the agent's benefit are both
genuinely weak/uncertain, not proven strengths - and the agent specifically now looks like a net
negative on two independent, larger samples (T041-B and AV01), not just an unproven positive.

**What we chose**: RAG + selective agent, frozen, retrieve -> rerank -> rule-boost ->
classify -> route -> selective agent.

**Why**: architectural principle (full-context's risk profile is expected to worsen on longer real
documents) rather than a clean accuracy win - the dev-sample, T041-B, and AV01 accuracy numbers have
all favored full-context or tied it with RAG, never clearly favoring the shipped RAG+agent design.

**What remains uncertain**: whether the long-document scalability principle holds up on real longer
documents (not started - the one hypothesis in this whole project that has never been tested at
all); whether the agent's negative effect on T041-B and AV01 would persist under a different routing
threshold or tool configuration (not tested - only the current frozen configuration has been
checked).

**What final evidence exists**: the official ContractNLI test-set evaluation (T041-B, historical,
caveats in Section 16) and the untouched architecture-validation run (AV01, Section 12b, complete)
both now point the same direction on the agent question - not resolved into a re-freeze here, but
no longer a single isolated data point either.